# Stage 4-1. Optimizers

이 노트북은 `src/core/optimizers.py`에 구현된 `SGD`와 `Adam` 옵티마이저를 실습한다.

| 클래스 | 업데이트 규칙 |
|---|---|
| `SGD` | $\theta \leftarrow \theta - \eta \nabla\theta$ |
| `Adam` | 1차·2차 모멘트 추정 + bias correction 적용 후 업데이트 |

**학습 목표**
1. `SGD`와 `Adam`의 인터페이스(`__init__`, `step`)를 확인한다.
2. 동일 학습률에서 SGD vs Adam 수렴 속도를 비교한다.
3. 학습률 민감도(lr sensitivity) 차이를 비교한다.
4. Adam의 모멘트 상태(`ms`, `vs`) 변화를 관찰한다.

## 0. 환경 설정

In [ ]:
import sys
sys.path.insert(0, "../..")

import numpy as np
import matplotlib.pyplot as plt

from src.models.mlp import MLP
from src.core.optimizers import SGD, Adam
from src.task import get_task_spec, transform_targets
from src.nn.losses import cross_entropy, cross_entropy_grad
from src.nn.metrics import accuracy

## 1. Optimizer 인터페이스 확인

In [ ]:
model = MLP(task="multiclass", seed=0)

# SGD
sgd = SGD(model, lr=0.01)
print("=== SGD ===")
print(f"lr     : {sgd.lr}")
print(f"params : {len(sgd.params)} 개 텐서")
print(f"grads  : {len(sgd.grads)} 개 텐서")
print(f"sgd.params[0] is model.params[0]: {sgd.params[0] is model.params[0]}")
print()

# Adam
adam = Adam(model, lr=0.001)
print("=== Adam ===")
print(f"lr    : {adam.lr}")
print(f"beta1 : {adam.beta1}")
print(f"beta2 : {adam.beta2}")
print(f"eps   : {adam.eps}")
print(f"ms (모멘트 버퍼) 수  : {len(adam.ms)}")
print(f"vs (분산 버퍼) 수    : {len(adam.vs)}")

## 2. 1 Step 업데이트 — SGD

In [ ]:
rng = np.random.default_rng(42)
N = 64
X = rng.normal(0, 1, (N, 784)).astype(np.float32)
L = rng.integers(0, 10, N).astype(np.uint8)
T = transform_targets(L, "multiclass")

model_sgd = MLP(task="multiclass", seed=0)
sgd_opt   = SGD(model_sgd, lr=0.01)

w_before = model_sgd.params[0].copy()

logits = model_sgd.forward(X)
model_sgd.backward(cross_entropy_grad(logits, T))
sgd_opt.step()

w_after = model_sgd.params[0]
delta   = w_before - w_after  # lr * grad

print("SGD 1 step 검증:")
print(f"  w 변화 최대값 : {np.abs(delta).max():.6f}")
print(f"  수식 검증 (lr*grad): {np.allclose(delta, 0.01 * model_sgd.grads[0], atol=1e-6)}")

## 3. SGD vs Adam — 수렴 속도 비교

In [ ]:
def run_training(opt_class, lr, epochs=60, seed=0):
    rng_t = np.random.default_rng(1)
    X_t = rng_t.normal(0, 1, (512, 784)).astype(np.float32)
    L_t = rng_t.integers(0, 10, 512).astype(np.uint8)
    T_t = transform_targets(L_t, "multiclass")

    model_t = MLP(task="multiclass", seed=seed)
    opt_t   = opt_class(model_t, lr=lr)

    losses, accs = [], []
    for _ in range(epochs):
        out   = model_t.forward(X_t)
        losses.append(float(cross_entropy(out, T_t)))
        accs.append(float(accuracy(out, T_t)))
        model_t.backward(cross_entropy_grad(out, T_t))
        opt_t.step()
    return losses, accs

# 동일 학습률로 비교 (Adam의 기본 lr=0.001이 SGD lr=0.01과 유사 스케일)
sgd_loss, sgd_acc   = run_training(SGD,  lr=0.05)
adam_loss, adam_acc = run_training(Adam, lr=0.001)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
fig.suptitle("SGD vs Adam — Loss / Accuracy", fontsize=13, fontweight="bold")

ax1.plot(sgd_loss,  label="SGD  (lr=0.05)",  color="steelblue",     linewidth=2)
ax1.plot(adam_loss, label="Adam (lr=0.001)", color="#E87B4C",       linewidth=2)
ax1.set_title("Cross-Entropy Loss")
ax1.set_xlabel("epoch")
ax1.set_ylabel("loss")
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(sgd_acc,  label="SGD  (lr=0.05)",  color="steelblue",     linewidth=2)
ax2.plot(adam_acc, label="Adam (lr=0.001)", color="#E87B4C",       linewidth=2)
ax2.set_title("Accuracy")
ax2.set_xlabel("epoch")
ax2.set_ylabel("accuracy")
ax2.set_ylim(0, 1)
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"SGD  최종 loss={sgd_loss[-1]:.4f}, acc={sgd_acc[-1]:.4f}")
print(f"Adam 최종 loss={adam_loss[-1]:.4f}, acc={adam_acc[-1]:.4f}")

## 4. 학습률 민감도 비교

In [ ]:
sgd_lrs  = [0.001, 0.01, 0.05, 0.1]
adam_lrs = [0.0001, 0.001, 0.005, 0.01]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Learning Rate Sensitivity", fontsize=13, fontweight="bold")

colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728"]

for lr, color in zip(sgd_lrs, colors):
    losses, _ = run_training(SGD, lr=lr)
    ax1.plot(losses, label=f"lr={lr}", color=color, linewidth=1.8)
ax1.set_title("SGD")
ax1.set_xlabel("epoch")
ax1.set_ylabel("loss")
ax1.set_ylim(0, 6)
ax1.legend()
ax1.grid(True, alpha=0.3)

for lr, color in zip(adam_lrs, colors):
    losses, _ = run_training(Adam, lr=lr)
    ax2.plot(losses, label=f"lr={lr}", color=color, linewidth=1.8)
ax2.set_title("Adam")
ax2.set_xlabel("epoch")
ax2.set_ylabel("loss")
ax2.set_ylim(0, 6)
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

> **관찰**: Adam은 학습률 범위가 넓어도 안정적으로 수렴한다. SGD는 lr이 너무 크면 발산하거나 진동한다.

## 5. Adam 모멘트 상태 관찰

In [ ]:
model_a = MLP(task="multiclass", seed=0)
adam_a  = Adam(model_a, lr=0.001)

rng_a = np.random.default_rng(0)
X_a = rng_a.normal(0, 1, (32, 784)).astype(np.float32)
L_a = rng_a.integers(0, 10, 32).astype(np.uint8)
T_a = transform_targets(L_a, "multiclass")

print("step 0 (초기):")
print(f"  iter    : {adam_a.iter}")
print(f"  m[0] 최대: {np.abs(adam_a.ms[0]).max():.6f}")
print(f"  v[0] 최대: {np.abs(adam_a.vs[0]).max():.6f}")

for step in [1, 5, 10]:
    while adam_a.iter < step:
        out_a = model_a.forward(X_a)
        model_a.backward(cross_entropy_grad(out_a, T_a))
        adam_a.step()

    print(f"\nstep {step}:")
    print(f"  iter    : {adam_a.iter}")
    print(f"  m[0] 최대: {np.abs(adam_a.ms[0]).max():.6f}")
    print(f"  v[0] 최대: {np.abs(adam_a.vs[0]).max():.6f}")
    # bias correction 적용 후 실제 학습률
    lr_eff = adam_a.lr / (1.0 - adam_a.beta2 ** adam_a.iter) ** 0.5 * (1.0 - adam_a.beta1 ** adam_a.iter)
    print(f"  실효 lr (bias-corrected): {lr_eff:.6f}")

> **Adam bias correction**: 초기 단계(iter가 작을 때)는 $m, v$ 추정치가 0 쪽으로 편향(bias)되어 있어 보정 항 $\frac{1}{1-\beta^t}$을 곱한다. 학습이 진행될수록 보정 항이 1에 수렴하여 사라진다.

## 6. Optimizer는 model 참조만 유지

In [ ]:
# optimizer.params와 model.params는 같은 리스트 원소를 참조한다
model_ref = MLP(task="multiclass", seed=0)
sgd_ref   = SGD(model_ref, lr=0.01)

print("sgd.params[0] is model.params[0] :", sgd_ref.params[0] is model_ref.params[0])
print("sgd.grads[0]  is model.grads[0]  :", sgd_ref.grads[0]  is model_ref.grads[0])
print()
print("→ optimizer.step()은 model 내부 배열을 in-place로 직접 수정한다.")

## 7. 정리

| 항목 | SGD | Adam |
|---|---|---|
| 업데이트 | $\theta -= lr \cdot g$ | 1차·2차 모멘트 + bias correction |
| 상태 | 없음 | `ms`, `vs`, `iter` |
| lr 민감도 | 높음 | 낮음 (적응적 학습률) |
| 기본 lr | 0.01–0.1 | 0.001 |

**인터페이스**
```python
opt = SGD(model, lr=0.01)   # 또는 Adam(model, lr=0.001)
# 매 step:
model.backward(grad)
opt.step()   # model.params in-place 업데이트, 반환값 없음
```

**핵심 설계 원칙**
- `opt.step()`은 반환값 없이 `model.params`를 in-place로 업데이트한다.
- `Trainer`는 `opt.step()` 한 줄로 어떤 optimizer도 교체 없이 사용할 수 있다.
- Stage 4에서는 `SGD`가 기본 optimizer이며, `Experiment` 설정으로 교체 가능하다.